In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time


In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

API_KEY = os.getenv("RAWG_API_KEY")

BASE_URL = "https://api.rawg.io/api/games"

In [ ]:
def fetch_games(pages=80, page_size=40):
    all_games = []
    
    for page in tqdm(range(1, pages + 1)):
        params = {
            "key": API_KEY,
            "page": page,
            "page_size": page_size
        }
        
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code != 200:
            print(f"Error en página {page}")
            continue
        
        data = response.json()
        all_games.extend(data["results"])
        
        time.sleep(1)
    
    return all_games

In [ ]:
games = fetch_games(pages=80)
len(games)

In [ ]:
def build_raw_dataframe(games):
    rows = []
    
    for g in games:
        row = {}
        
        row["id"] = g.get("id")
        row["name"] = g.get("name")
        row["slug"] = g.get("slug")
        
        row["released"] = g.get("released")
        
        row["rating"] = g.get("rating")
        row["ratings_count"] = g.get("ratings_count")
        row["reviews_text_count"] = g.get("reviews_text_count")
        row["metacritic"] = g.get("metacritic")
        row["added"] = g.get("added")
        row["suggestions_count"] = g.get("suggestions_count")
        
        genres = [genre["name"] for genre in g.get("genres", [])]
        row["genres"] = ", ".join(genres)
        
        platforms = [p["platform"]["name"] for p in g.get("platforms", [])]
        row["platforms"] = ", ".join(platforms)
        
        devs = [d["name"] for d in g.get("developers", [])]
        row["developers"] = ", ".join(devs)
        
        pubs = [p["name"] for p in g.get("publishers", [])]
        row["publishers"] = ", ".join(pubs)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [ ]:
df_raw = build_raw_dataframe(games)
df_raw.head()

In [ ]:
df_raw = df_raw.drop_duplicates(subset="id")
df_raw = df_raw.reset_index(drop=True)

In [ ]:
df_raw.to_csv("rawg_games_full_dataset.csv", index=False)